# Automated Vehicle Damage Triage for Insurance Claims  

datset: [VehiDE Dataset](https://www.kaggle.com/datasets/hendrichscullen/vehide-dataset-automatic-vehicle-damage-detection)  
**Stage 1**: Damage presence / type classifier

* Predict whether the image shows damage, and if labels allow, classify broad damage categories such as dent, scratch, crack, broken part, or severity bucket.

**Stage 2:** Damage localization

* Add bounding-box or segmentation-based localization if annotations exist, because ControlExpert publicly highlights vehicle-part segmentation and precise damage detection rather than image-level labels only.

## 1-Data ingestion

In [ ]:
# # Download data
# import kagglehub

# # Download latest version
# path = kagglehub.dataset_download("hendrichscullen/vehide-dataset-automatic-vehicle-damage-detection")

# print("Path to dataset files:", path)

In [ ]:
from utils import walk_through_dir

walk_through_dir("VehiDe_dataset")

In [ ]:
from utils import plot_random_images_with_labels

plot_random_images_with_labels(
    json_path="VehiDe_dataset/0Train_via_annos.json",
    images_dir="VehiDe_dataset/image/image",
    num_samples=6,
    n_cols=3)

In [ ]:
from utils import get_unique_labels

train_labels = get_unique_labels("VehiDe_dataset/0Train_via_annos.json")
val_labels   = get_unique_labels("VehiDe_dataset/0Val_via_annos.json")

In [ ]:
from utils import plot_label_distribution

plot_label_distribution(
    train_json="VehiDe_dataset/0Train_via_annos.json",
    val_json="VehiDe_dataset/0Val_via_annos.json")

In [ ]:
from PIL import Image
import numpy as np
from pathlib import Path
from collections import Counter

images_dir = Path("VehiDe_dataset/image/image")

widths, heights, modes = [], [], []
for img_path in images_dir.glob("*.jpg"):
    img = Image.open(img_path)
    w, h = img.size
    widths.append(w)
    heights.append(h)
    modes.append(img.mode)

print(f"Total images checked: {len(widths)}")
print(f"Width  — min: {min(widths)}, max: {max(widths)}, mean: {np.mean(widths):.0f}")
print(f"Height — min: {min(heights)}, max: {max(heights)}, mean: {np.mean(heights):.0f}")
print(f"Color modes: {Counter(modes)}")

In [ ]:
import numpy as np
from PIL import Image
from pathlib import Path

img_path = next(Path("VehiDe_dataset/image/image").glob("*.jpg"))
img = Image.open(img_path).convert("RGB")
arr = np.array(img)

print(f"dtype:  {arr.dtype}")
print(f"min:    {arr.min()}")
print(f"max:    {arr.max()}")
print(f"shape:  {arr.shape}")

## 2- base_model

In [ ]:
# Convert this VehiDe unstructure to YOLO structure format Also translate labels to english
import json
import shutil
from pathlib import Path

# ── config ──────────────────────────────────────────────
SRC_IMAGES_TRAIN = Path("VehiDe_dataset/image/image")
SRC_IMAGES_VAL   = Path("VehiDe_dataset/validation/validation")
TRAIN_JSON       = Path("VehiDe_dataset/0Train_via_annos.json")
VAL_JSON         = Path("VehiDe_dataset/0Val_via_annos.json")
OUT_DIR          = Path("yolo_dataset")
# ────────────────────────────────────────────────────────

LABEL_MAP = {
    "tray_son":    "paint_scratch",
    "mop_lom":     "dented",
    "rach":        "crack",
    "mat_bo_phan": "missing_parts",
    "be_den":      "crushed_panel",
    "thung":       "punctured",
    "vo_kinh":     "broken_glass",
}

def get_all_classes(*json_paths):
    classes = set()
    for jp in json_paths:
        with open(jp) as f:
            data = json.load(f)
        for entry in data.values():
            for region in entry["regions"]:
                eng = LABEL_MAP.get(region["class"], region["class"])
                classes.add(eng)
    return sorted(classes)

def convert_split(json_path, src_images_dir, split):
    with open(json_path) as f:
        data = json.load(f)

    img_out = OUT_DIR / "images" / split
    lbl_out = OUT_DIR / "labels" / split
    img_out.mkdir(parents=True, exist_ok=True)
    lbl_out.mkdir(parents=True, exist_ok=True)

    from PIL import Image  # import once here

    converted = 0
    for fname, entry in data.items():
        src = src_images_dir / fname
        if not src.exists():
            print(f"Missing: {src}")
            continue

        dst_img = img_out / fname
        dst_lbl = lbl_out / (Path(fname).stem + ".txt")

        # If both image and label already exist, skip this file
        if dst_img.exists() and dst_lbl.exists():
            continue

        # copy image if needed
        if not dst_img.exists():
            shutil.copy(src, dst_img)

        # get image size for normalization
        w, h = Image.open(src).size

        # write / overwrite label file
        lines = []
        for region in entry["regions"]:
            raw_class = region["class"]
            eng_class = LABEL_MAP.get(raw_class, raw_class)
            cls_id = class_to_id[eng_class]

            xs = [x / w for x in region["all_x"]]
            ys = [y / h for y in region["all_y"]]
            coords = [f"{x:.6f} {y:.6f}" for x, y in zip(xs, ys)]
            lines.append(f"{cls_id} " + " ".join(coords))

        dst_lbl.write_text("\n".join(lines))
        converted += 1

    print(f"[{split}] processed entries: {len(data)}, newly converted: {converted}")

# ── idempotent guard ────────────────────────────────────
dataset_yaml = OUT_DIR / "dataset.yaml"
if dataset_yaml.exists():
    print(f"{dataset_yaml} already exists — skipping conversion. "
          f"Delete it (and/or yolo_dataset) if you want to regenerate.")
else:
    # build class map
    all_classes = get_all_classes(TRAIN_JSON, VAL_JSON)
    class_to_id = {cls: i for i, cls in enumerate(all_classes)}
    print("Classes:", class_to_id)

    # convert both splits
    convert_split(TRAIN_JSON, SRC_IMAGES_TRAIN, "train")
    convert_split(VAL_JSON,   SRC_IMAGES_VAL,   "val")

    # write dataset.yaml
    yaml_content = f"""path: {OUT_DIR.resolve()}
train: images/train
val: images/val

nc: {len(all_classes)}
names: {all_classes}
"""
    dataset_yaml.write_text(yaml_content)
    print("dataset.yaml written")
    print("All done!")

In [ ]:
# !yolo settings tensorboard=True

In [ ]:
# !nvidia-smi

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")

In [ ]:
from ultralytics import YOLO

# segmentation model, not detection
model = YOLO("yolo11n-seg.pt")   # or "yolov8n-seg.pt" depending on version

model.train(
    data="yolo_dataset/dataset.yaml",
    epochs=100,
    imgsz=640,
    batch=32,
    patience=5,
    device=0,
    workers=8,
    cache=True,
    project="runs/vehide",
    name="yolo11n-seg-vehide",
)